# Solving Question 5 and 6 with Pandas

## Question 5
Actually it's pretty clear from experience as well previous looks at the spans
that `llm` will take significantly longer, but I'll still show it.

In [1]:
import sqlite3
import pandas as pd

In [2]:
# get the data

# connect to sqlite db
con = sqlite3.connect("traces.db")

# read data to pandas
df = pd.read_sql("SELECT * FROM spans", con)

# close db connection again
con.close()

In [3]:
# check data
df

,name,start_time,end_time,input_tokens,output_tokens,cost
0,search,1784164082892290000,1784164082893919000,NaN,NaN,NaN
1,llm,1784164082896398000,1784164088591899000,8183.0,399.0,0.010178
2,rag,1784164082892248000,1784164088593185000,NaN,NaN,NaN
3,search,1784164568055880000,1784164568056915000,NaN,NaN,NaN
4,llm,1784164568057796000,1784164577540063000,8183.0,417.0,0.010268
5,rag,1784164568055850000,1784164577541095000,NaN,NaN,NaN
6,search,1784166609536599000,1784166609537941000,NaN,NaN,NaN
7,llm,1784166609539815000,1784166614986428000,8183.0,469.0,0.010528
8,rag,1784166609536570000,1784166614987655000,NaN,NaN,NaN
9,search,1784166621984327000,1784166621985397000,NaN,NaN,NaN


In [4]:
# question is about duration of llm vs search, so exclude rag
# also select relevant columns only (tokens etc. don't matter for this question)
df_times = df[df["name"] != "rag"][["name", "start_time", "end_time"]].copy()

# compute duration

# nanoseconds (as used by open tel)
df_times["duration_ns"] = df_times["end_time"] - df_times["start_time"]
# seconds (rounded)
df_times["duration_s"] = df_times["duration_ns"] / 1_000_000_000

# check result
df_times

,name,start_time,end_time,duration_ns,duration_s
0,search,1784164082892290000,1784164082893919000,1629000,0.001629
1,llm,1784164082896398000,1784164088591899000,5695501000,5.695501
3,search,1784164568055880000,1784164568056915000,1035000,0.001035
4,llm,1784164568057796000,1784164577540063000,9482267000,9.482267
6,search,1784166609536599000,1784166609537941000,1342000,0.001342
7,llm,1784166609539815000,1784166614986428000,5446613000,5.446613
9,search,1784166621984327000,1784166621985397000,1070000,0.001070
10,llm,1784166621986379000,1784166627746701000,5760322000,5.760322


Already now it's clear, but let's finish this now.
I think with more data it would really not be less clear at this point, but to
be absolutely sure, you'd have to do the computation.

In [5]:
# group by name and compute mean
df_times = df_times.groupby("name").agg(mean_duration=("duration_s", "mean"))

# check result
df_times

,mean_duration
name,
llm,6.596176
search,0.001269


There we have it.
llm takes **much** longer, and we showed it.

## Question 6

In [8]:
input_tokens = df[df["name"] == "llm"]["input_tokens"]
input_tokens

1     8183.0
4     8183.0
7     8183.0
10    8183.0
Name: input_tokens, dtype: float64

They are the same, so the retrieval always gets the same context.
Since we're using a text search with a basic implementation from our own
indexed data, this is expected.